In [1]:
# Setup
import chi, os, time, datetime
from chi import lease, server, network, context

context.version = "1.0"
context.choose_project()   
context.choose_site(default="KVM@TACC")
print("Setup done")

Setup done


In [2]:
# Names
suffix      = "proj19"
lease_name  = f"bestshot-k8s-{suffix}"
node_name   = f"node-bestshot-{suffix}"
volume_name = f"bestshot-vol-{suffix}"

print(f"Lease name:  {lease_name}")
print(f"Node name:   {node_name}")
print(f"Volume name: {volume_name}")

Lease name:  bestshot-k8s-proj19
Node name:   node-bestshot-proj19
Volume name: bestshot-vol-proj19


In [3]:
# Create lease
l = lease.Lease(
    lease_name,
    duration=datetime.timedelta(days=1)
)

l.add_flavor_reservation(
    id=chi.server.get_flavor_id("m1.medium"),
    amount=1
)

l.submit(idempotent=True)
print("Lease submitted! Waiting...")
l.show()

Waiting for lease to start...


Lease bestshot-k8s-proj19 has reached status active
Lease submitted! Waiting...


HTML(value='\n        <h2>Lease Details</h2>\n        <table>\n            <tr><th>Name</th><td>bestshot-k8s-p…

Lease Details:
Name: bestshot-k8s-proj19
ID: b92a0445-f7d9-47d5-8215-4fe3a755f411
Status: ACTIVE
Start Date: 2026-04-05 06:38:00
End Date: 2026-04-06 06:38:00
User ID: e85bd4460d9b1b01d909d38aace9700d34d4ea2dbee28bcf76216e4ae058d896
Project ID: 89f528973fea4b3a981f9b2344e522de

Node Reservations:

Floating IP Reservations:

Network Reservations:

Flavor Reservations:
ID: e63f075c-44ee-4311-9119-4d81f0863262, Status: active, Flavor: e63f075c-44ee-4311-9119-4d81f0863262, Amount: 1

Events:


In [4]:
# Launch VMs
flavor = l.get_reserved_flavors()[0].name
print(f"Using flavor: {flavor}")

node = server.Server(
    node_name,
    image_name="CC-Ubuntu24.04",
    flavor_name=flavor
)
node.submit(idempotent=True)
print("Waiting for node to boot...")
node.wait()
print("Node is ready!")

Using flavor: reservation:e63f075c-44ee-4311-9119-4d81f0863262


The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.


Waiting for server node-bestshot-proj19's status to become ACTIVE. This typically takes 10 minutes for baremetal, but can take up to 20 minutes.


Server has moved to status ACTIVE


Attribute,node-bestshot-proj19
Id,95c11c2c-8397-4bf4-b8f5-8a58908c9892
Status,ACTIVE
Image Name,CC-Ubuntu24.04
Flavor Name,reservation:e63f075c-44ee-4311-9119-4d81f0863262
Addresses,sharednet1: IP: 10.56.1.116 (v4) Type: fixed MAC: fa:16:3e:ab:ac:11
Network Name,sharednet1
Created At,2026-04-05T06:39:15Z
Keypair,dj2680_nyu_edu-jupyter
Reservation Id,None
Host Id,6a2e58af90e692c7aab1e7ff45e4afa140c81c6684b0e761fc8d3349


Waiting for node to boot...
Waiting for server node-bestshot-proj19's status to become ACTIVE. This typically takes 10 minutes for baremetal, but can take up to 20 minutes.


Server has moved to status ACTIVE
Node is ready!


In [5]:
# Assign floating IP
node.associate_floating_ip()
node.refresh()
fip = node.get_floating_ip()
print(f"Floating IP: {fip}")
print(f"SSH command: ssh -i ~/.ssh/id_rsa_chameleon cc@{fip}")

The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.


Floating IP: 129.114.27.200
SSH command: ssh -i ~/.ssh/id_rsa_chameleon cc@129.114.27.200


In [6]:
# Security groups
security_groups = [
    ("allow-ssh",   22,    "SSH access"),
    ("allow-http",  80,    "HTTP access"),
    ("allow-30283", 30283, "Immich NodePort"),
    ("allow-30500", 30500, "MLflow NodePort"),
    ("allow-6443",  6443,  "K8S API server"),
]

for sg_name, port, desc in security_groups:
    sg = network.SecurityGroup({"name": sg_name, "description": desc})
    sg.add_rule("ingress", "tcp", port)
    sg.submit(idempotent=True)
    node.add_security_group(sg_name)
    print(f"Added: {sg_name} (port {port})")

The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.
The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.


Added: allow-ssh (port 22)


The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.


Added: allow-http (port 80)


The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.


Added: allow-30283 (port 30283)


The python binding code in neutronclient is deprecated in favor of OpenstackSDK, please use that as this will be removed in a future release.


Added: allow-30500 (port 30500)
Added: allow-6443 (port 6443)


In [7]:
# Block volume
conn = chi.connection()

vol = conn.block_storage.create_volume(
    name=volume_name,
    size=20
)
print(f"Volume created: {volume_name}")
print("Waiting for volume to be ready...")

for i in range(24):
    vol = conn.block_storage.get_volume(vol.id)
    print(f"Volume status: {vol.status}")
    if vol.status == "available":
        print("Volume ready!")
        break
    time.sleep(10)

conn.compute.create_volume_attachment(
    node.id,
    volumeId=vol.id
)
print(f"Volume attached to node!")

Volume created: bestshot-vol-proj19
Waiting for volume to be ready...
Volume status: creating
Volume status: available
Volume ready!
Volume attached to node!


In [8]:
# Summary
node.refresh()
fip = node.get_floating_ip()

print(f"\n{'='*55}")
print(f"  BESTSHOT INFRASTRUCTURE READY")
print(f"{'='*55}")
print(f"  Node:        {node_name}")
print(f"  Floating IP: {fip}")
print(f"  Volume:      {volume_name} (20GB)")
print(f"{'='*55}")
print(f"\n  SSH into your node:")
print(f"  ssh -i ~/.ssh/id_rsa_chameleon cc@{fip}")
print(f"\n  Immich:  http://{fip}:30283")
print(f"  MLflow:  http://{fip}:30500")
print(f"{'='*55}")


  BESTSHOT INFRASTRUCTURE READY
  Node:        node-bestshot-proj19
  Floating IP: 129.114.27.200
  Volume:      bestshot-vol-proj19 (20GB)

  SSH into your node:
  ssh -i ~/.ssh/id_rsa_chameleon cc@129.114.27.200

  Immich:  http://129.114.27.200:30283
  MLflow:  http://129.114.27.200:30500


In [ ]:
l = lease.get_lease("bestshot-k8s-proj19")
l.show()

# Delete the lease
l.delete()
print("Lease deleted. VM and resources are terminated.")